In [ ]:
#task 1

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt             #importing libraries and loading the csv
df_original = pd.DataFrame(pd.read_csv("../data/raw.csv"))          #i had a problem using relative path and copilot told me to do add ../ in the beginning 
df = df_original.copy()     #i made a copy so if anything goes wrong it doesnt get ruined

print(df.head())

In [ ]:
def clean_data(dataframe):
    #summerise the content of the dataframe
    print("-----------------------------------------------------------------")
    print(dataframe.describe())
    print("-----------------------------------------------------------------")
    print(dataframe.info())
    print("-----------------------------------------------------------------")
    rows = int((dataframe.shape)[0])
    columns = int((dataframe.shape)[1])
    print(f"rows: {rows}, columns: {columns}")   #to make it look more simple
    print("-----------------------------------------------------------------")
    print(dataframe.dtypes)
    print("-----------------------------------------------------------------")
    missing_list = dataframe.isnull().sum().sort_values(ascending = False).to_dict()                #find the missing values and decide what to do with them
    print("missing from each row:\n", missing_list)   #this shows the columns with missing values
    print("-----------------------------------------------------------------")
    dataframe.drop(columns=['Cabin'], inplace=True)    #i have decided to drop the (cabin) column because it has too many missing valuess to fill and is an expendable value in my oppinion
    dataframe['Age'] = dataframe['Age'].fillna(dataframe['Age'].mean())    #(age) is an important feature and doesnt have as many missing values as cabin so i decided to fill using mean since it is a numerical value
    dataframe['Embarked'] = dataframe['Embarked'].fillna(dataframe['Embarked'].mode()[0])     #since (embarked) is an object value and only missing 2 values i decided to use mode to fill
    dataframe["Embarked"] = dataframe["Embarked"].astype("category")

    print("duplicates in dataframe:", dataframe.duplicated().sum())
    #i will use the iqr method to find outliers
    
    plt.boxplot(dataframe['Age'])
    plt.title("age boxplot")        #show the bloxplot without finding outliers
    plt.show()

    q1 = dataframe["Age"].quantile(0.25)        #get the first and third quarter
    q3 = dataframe["Age"].quantile(0.75)           
    
    iqr = q3 -q1
    lower = q1 - 1.5*iqr                #IQR method
    upper = q3 + 1.5*iqr
    outliers = (dataframe["Age"] < lower) | (dataframe["Age"] > upper)
    up_cap = dataframe["Age"].quantile(0.99)
    dataframe["Age"] = dataframe["Age"].clip(upper = up_cap)
    
    plt.boxplot(dataframe['Age'])
    plt.title("age capped boxplot")         #to show the differance between before and after removing outliers
    plt.show()

    



    print("-----------------------------------------------------------------")
    print("info after modification")
    print("-----------------------------------------------------------------")
    print(dataframe.describe())
    print("-----------------------------------------------------------------")
    print(dataframe.info())
    print("-----------------------------------------------------------------")
    rows = int((dataframe.shape)[0])
    columns = int((dataframe.shape)[1])
    print(f"rows: {rows}, columns: {columns}")   
    print("-----------------------------------------------------------------")
    print("-----------------------------------------------------------------")
    missing_list = dataframe.isnull().sum().sort_values(ascending = False).to_dict()
    print("missing from each row:\n", missing_list)
    return dataframe

df = clean_data(df)


In [ ]:
    #first check, no missing values in key columns
assert df['Age'].isnull().sum() == 0, "age still has missing values"
assert df['Embarked'].isnull().sum() == 0, "embarked still has missing values"

#second check, are target values valid
assert df['Survived'].isin([0, 1]).all(), "there are invalid values in survived"    

#third check, is the shape of the dataframe as expected
expected_columns = 11
assert df.shape[1] == expected_columns, "column amount is incorrect"



df.to_csv("../data/cleaned/clean data.csv", index=False)        #make a new clean data file to use in other tasks